# Lesson 01: Course Setup — Google AI Studio Free Tier & Open-Source Agent Stack

In this notebook, you will:
1. Install the open-source AI agent stack (`google-genai`, `langgraph`, `chromadb`, `pydantic`, `python-dotenv`).
2. Load your free `GEMINI_API_KEY` from **Google Colab Secrets** (with an automatic offline simulation fallback if you haven't added a key yet).
3. Test your first tool-enabled Gemini model and an in-memory vector search index.

In [ ]:
!pip install -q -U google-genai langgraph chromadb pydantic python-dotenv

In [ ]:
import os
from google import genai

def get_gemini_client():
    api_key = None
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass
    if not api_key:
        try:
            from dotenv import load_dotenv
            load_dotenv()
        except Exception:
            pass
        api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        print("GEMINI_API_KEY not found. Running in offline simulation mode.")
        return None
    return genai.Client(api_key=api_key)

client = get_gemini_client()
print("Live Gemini client ready:", client is not None)

In [ ]:
if client:
    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents="In one sentence, explain what makes an AI system an agent."
    )
    print(resp.text)
else:
    print("[Simulated Gemini 2.5 Flash] An AI agent pairs an LLM with tools, memory, and an action loop to achieve goals in an environment.")

In [ ]:
import chromadb

chroma = chromadb.Client()
collection = chroma.get_or_create_collection("course_setup_kb")
collection.upsert(
    ids=["setup-1", "setup-2"],
    documents=[
        "Store GEMINI_API_KEY in Google Colab Secrets for zero-cost free tier access.",
        "Use Ollama at http://localhost:11434/v1 for 100% offline local Qwen2.5 agents."
    ]
)
res = collection.query(query_texts=["How do I run offline local agents?"], n_results=1)
print("Top Vector Search Hit:", res["documents"][0][0])